# Guardrailer — Dual-GPU Enhanced RAG Ingestion

Pre-computes all multi-signal scoring features for the Guardrailer RAG system using **both T4 GPUs**.

**Run on Kaggle with 2x T4 GPU accelerator and Internet ON.**

**Resumable:** Checkpoints auto-save after each phase. If session dies, just re-run all cells — completed phases are skipped.

### First run
Just run all cells. No setup needed.

### On restart (same session or new session)
Just run all cells. Completed phases load from checkpoint, pending phases recompute.

Pipeline phases:
1. Load dataset
2. Compute IDF corpus statistics
3. Dense embeddings (sequential dual-GPU)
4. Category centroids
5. k-Means clustering (FAISS-GPU)
6. k-NN graph (FAISS-GPU)
7. Uniqueness scores
8. Cross-encoder pre-scoring (sequential dual-GPU)
9. Export all pre-computed files

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

CFG = {
    "seed": 42,
    "dense_model": "BAAI/bge-large-en-v1.5",
    "cross_encoder_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "n_clusters": 128,
    "n_neighbors": 10,
    "embed_batch_size": 256,
    "ce_batch_size": 128,
    "out_dir": "/kaggle/working/guardrailer_output",
    "ckpt_dir": "/kaggle/working/guardrailer_output/.checkpoints",
    "ckpt_dataset": "guardrailer-checkpoints",
    "faiss_temp_memory_mb": 512,
    "max_text_len": 400,
}

os.makedirs(CFG["out_dir"], exist_ok=True)
os.makedirs(CFG["ckpt_dir"], exist_ok=True)
print(f"Output: {CFG['out_dir']}")
print(f"Checkpoints: {CFG['ckpt_dir']}")

In [ ]:
!pip install -q sentence-transformers faiss-gpu scikit-learn pandas pyarrow kaggle

In [ ]:
import os, json, math, time, gc, logging, signal, atexit, sys, subprocess, tarfile
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

# ---------------------------------------------------------------------------
# GPU utilities
# ---------------------------------------------------------------------------

def gpu_memory_status(gpu_id=None):
    if not torch.cuda.is_available():
        return
    devices = [gpu_id] if gpu_id is not None else range(torch.cuda.device_count())
    for i in devices:
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {allocated:.2f} GB alloc / {reserved:.2f} GB reserved / {total:.1f} GB total")


def clear_gpu():
    if not torch.cuda.is_available():
        return
    for i in range(torch.cuda.device_count()):
        torch.cuda.empty_cache()
    gc.collect()


# ---------------------------------------------------------------------------
# Checkpoint system
# ---------------------------------------------------------------------------

CKPT_DIR = CFG["ckpt_dir"]


def _get_kaggle_username():
    """Auto-detect Kaggle username from CLI config."""
    try:
        result = subprocess.run(
            ["kaggle", "config", "path"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            # Output is the config dir path, e.g. /root/.kaggle
            # We need the username from kaggle.json
            config_dir = result.stdout.strip()
            kaggle_json = os.path.join(config_dir, "kaggle.json")
            if os.path.exists(kaggle_json):
                with open(kaggle_json) as f:
                    creds = json.load(f)
                    return creds.get("username", "")
    except Exception:
        pass
    # Fallback: try environment variable
    return os.environ.get("KAGGLE_USERNAME", "")


KAGGLE_USER = _get_kaggle_username()
CKPT_SLUG = f"{KAGGLE_USER}/{CFG['ckpt_dataset']}" if KAGGLE_USER else ""

if not KAGGLE_USER:
    print("[WARN] Could not detect Kaggle username. Checkpoint persistence disabled.")
    print("       Checkpoints will only survive within this session.")
else:
    print(f"[INFO] Kaggle user: {KAGGLE_USER}")
    print(f"[INFO] Checkpoint dataset: {CKPT_SLUG}")


def _phase_marker(phase_name):
    return os.path.join(CKPT_DIR, f"phase_{phase_name}.done")


def is_phase_done(phase_name):
    return os.path.exists(_phase_marker(phase_name))


def mark_phase_done(phase_name):
    marker = _phase_marker(phase_name)
    with open(marker, "w") as f:
        json.dump({"phase": phase_name, "time": time.time()}, f)
    print(f"  [CHECKPOINT] Phase '{phase_name}' marked complete.")


def save_json_checkpoint(name, data):
    path = os.path.join(CKPT_DIR, f"{name}.json")
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, path)
    print(f"  [CHECKPOINT] Saved {name}.json ({os.path.getsize(path) / 1024:.0f} KB)")


def save_npy_checkpoint(name, arr):
    path = os.path.join(CKPT_DIR, f"{name}.npy")
    tmp = path + ".tmp"
    np.save(tmp, arr)
    os.replace(tmp + ".npy", path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  [CHECKPOINT] Saved {name}.npy ({size_mb:.1f} MB)")


def save_parquet_checkpoint(name, df):
    path = os.path.join(CKPT_DIR, f"{name}.parquet")
    tmp = path + ".tmp"
    df.to_parquet(tmp, engine="pyarrow", index=False)
    os.replace(tmp, path)
    print(f"  [CHECKPOINT] Saved {name}.parquet ({os.path.getsize(path) / 1e6:.1f} MB)")


def load_json_checkpoint(name):
    path = os.path.join(CKPT_DIR, f"{name}.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def load_npy_checkpoint(name):
    path = os.path.join(CKPT_DIR, f"{name}.npy")
    if not os.path.exists(path):
        return None
    return np.load(path)


def load_parquet_checkpoint(name):
    path = os.path.join(CKPT_DIR, f"{name}.parquet")
    if not os.path.exists(path):
        return None
    return pd.read_parquet(path)


# ---------------------------------------------------------------------------
# Kaggle Dataset persistence (auto-creates dataset if needed)
# ---------------------------------------------------------------------------

def _ensure_dataset_exists():
    """Create the checkpoint dataset on Kaggle if it doesn't exist."""
    if not KAGGLE_USER or not CKPT_SLUG:
        return False

    # Check if dataset exists
    result = subprocess.run(
        ["kaggle", "datasets", "list", "-d", CKPT_SLUG],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0 and "No datasets found" not in result.stdout:
        return True  # Dataset exists

    # Create the dataset
    print(f"  [PERSIST] Creating dataset {CKPT_SLUG} ...")
    # Create a minimal metadata file
    meta_dir = os.path.join(CFG["out_dir"], "_dataset_meta")
    os.makedirs(meta_dir, exist_ok=True)
    with open(os.path.join(meta_dir, "dataset-metadata.json"), "w") as f:
        json.dump({
            "title": CFG["ckpt_dataset"],
            "id": CKPT_SLUG,
            "licenses": [{"name": "CC0-1.0"}]
        }, f, indent=2)

    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", meta_dir, "-t", CKPT_SLUG],
        capture_output=True, text=True, timeout=120
    )
    # Cleanup
    import shutil
    shutil.rmtree(meta_dir, ignore_errors=True)

    if result.returncode == 0:
        print(f"  [PERSIST] Dataset created: {CKPT_SLUG}")
        return True
    else:
        print(f"  [PERSIST] Dataset creation failed (non-fatal): {result.stderr[:200]}")
        return False


def upload_checkpoints():
    """Tar checkpoint dir and push to Kaggle Dataset."""
    if not KAGGLE_USER or not CKPT_SLUG:
        return

    ckpt_files = os.listdir(CKPT_DIR)
    if not ckpt_files:
        return

    tar_path = os.path.join(CFG["out_dir"], "checkpoints.tar.gz")
    print(f"  [PERSIST] Packing {len(ckpt_files)} checkpoint files ...")
    with tarfile.open(tar_path, "w:gz") as tar:
        for fname in sorted(ckpt_files):
            tar.add(os.path.join(CKPT_DIR, fname), arcname=fname)

    tar_size = os.path.getsize(tar_path) / 1e6
    print(f"  [PERSIST] Tarball: {tar_size:.1f} MB")

    try:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", CFG["out_dir"],
             "-m", f"checkpoint update {time.strftime('%Y-%m-%d %H:%M:%S')}",
             "-t", CKPT_SLUG],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode == 0:
            print(f"  [PERSIST] Uploaded to {CKPT_SLUG}")
        else:
            print(f"  [PERSIST] Upload failed (non-fatal): {result.stderr[:150]}")
    except Exception as e:
        print(f"  [PERSIST] Upload error (non-fatal): {e}")

    if os.path.exists(tar_path):
        os.remove(tar_path)


def download_checkpoints():
    """Download checkpoint tarball from Kaggle Dataset and extract."""
    existing = [f for f in os.listdir(CKPT_DIR) if not f.startswith("_")]
    if existing:
        print(f"  [PERSIST] {len(existing)} local checkpoint files found — skipping download.")
        return True

    if not KAGGLE_USER or not CKPT_SLUG:
        print("  [PERSIST] No Kaggle credentials — starting from scratch.")
        return False

    print(f"  [PERSIST] Downloading from {CKPT_SLUG} ...")
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", CKPT_SLUG,
             "-p", CFG["out_dir"], "--unzip"],
            capture_output=True, text=True, timeout=600
        )
        if result.returncode != 0:
            print(f"  [PERSIST] Download failed (starting fresh): {result.stderr[:150]}")
            return False

        tar_path = os.path.join(CFG["out_dir"], "checkpoints.tar.gz")
        if os.path.exists(tar_path):
            print(f"  [PERSIST] Extracting ...")
            with tarfile.open(tar_path, "r:gz") as tar:
                tar.extractall(path=CKPT_DIR)
            os.remove(tar_path)
            n_files = len([f for f in os.listdir(CKPT_DIR) if not f.startswith("_")])
            print(f"  [PERSIST] Extracted {n_files} checkpoint files.")
            return True
        else:
            # Files might be directly in output dir
            for f in os.listdir(CFG["out_dir"]):
                if f.endswith((".done", ".npy", ".json", ".parquet")):
                    src = os.path.join(CFG["out_dir"], f)
                    dst = os.path.join(CKPT_DIR, f)
                    if not os.path.exists(dst):
                        os.rename(src, dst)
            n_files = len([f for f in os.listdir(CKPT_DIR) if not f.startswith("_")])
            if n_files:
                print(f"  [PERSIST] Found {n_files} checkpoint files.")
                return True
            print("  [PERSIST] No checkpoints found — starting fresh.")
            return False
    except Exception as e:
        print(f"  [PERSIST] Download error: {e}")
        return False


# ---------------------------------------------------------------------------
# Graceful shutdown watchdog
# ---------------------------------------------------------------------------

_pending_checkpoints = []

def _flush_checkpoints_on_exit():
    if not _pending_checkpoints:
        return
    print("\n  [WATCHDOG] Flushing pending checkpoints before exit ...")
    for name, data, dtype in _pending_checkpoints:
        try:
            if dtype == "npy":
                save_npy_checkpoint(name, data)
            elif dtype == "parquet":
                save_parquet_checkpoint(name, data)
            elif dtype == "json":
                save_json_checkpoint(name, data)
        except Exception as e:
            print(f"    [WATCHDOG] Failed to save {name}: {e}")
    _pending_checkpoints.clear()
    try:
        upload_checkpoints()
    except Exception as e:
        print(f"    [WATCHDOG] Upload on exit failed: {e}")
    print("  [WATCHDOG] Pending checkpoints flushed.")


def _signal_handler(signum, frame):
    sig_name = signal.Signals(signum).name
    print(f"\n  [WATCHDOG] Received {sig_name} — saving state ...")
    _flush_checkpoints_on_exit()
    sys.exit(0)


signal.signal(signal.SIGTERM, _signal_handler)
signal.signal(signal.SIGINT, _signal_handler)
atexit.register(_flush_checkpoints_on_exit)


# ---------------------------------------------------------------------------
# Init: ensure dataset, download checkpoints, show status
# ---------------------------------------------------------------------------

_ensure_dataset_exists()
download_checkpoints()

num_gpus = torch.cuda.device_count()
print(f"\nAvailable GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} \u2014 {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

PHASE_NAMES = ["dataset", "idf", "embeddings", "centroids", "kmeans", "knn", "uniqueness", "crossencoder", "export"]
print(f"\nCheckpoint status:")
for pn in PHASE_NAMES:
    status = "DONE" if is_phase_done(pn) else "pending"
    print(f"  {pn:<20s} {status}")
completed = sum(1 for pn in PHASE_NAMES if is_phase_done(pn))
print(f"\n  {completed}/{len(PHASE_NAMES)} phases completed")

---
## Phase 1: Load Dataset

In [ ]:
PARQUET_PATH = "/kaggle/input/datasets/prashannadeveloper/dataset/unified_security_dataset.parquet"

if is_phase_done("dataset"):
    print("[RESUME] Loading dataset from checkpoint ...")
    df = load_parquet_checkpoint("dataset")
else:
    df = pd.read_parquet(PARQUET_PATH)
    save_parquet_checkpoint("dataset", df)
    mark_phase_done("dataset")
    upload_checkpoints()

print(f"Dataset: {len(df):,} samples")
print(f"  Malicious: {int(df['is_malicious'].sum()):,}")
print(f"  Benign:    {int((~df['is_malicious']).sum()):,}")
print(f"\nCategories:")
for cat, cnt in df["attack_category"].value_counts().items():
    print(f"  {cat:<35s} {cnt:>8,}")

---
## Phase 2: Corpus Statistics

In [ ]:
SPARSE_KEYWORDS = [
    "ignore previous", "override", "bypass", "jailbreak", "system prompt",
    "your instructions", "forget", "disregard", "dan", "do anything now",
    "act as", "roleplay", "pretend you", "hypothetical", "in theory",
    "markdown injection", "code comment", "readme", "yaml", "json payload",
    "<script>", "]]>", "```", "<!--", "-->", "eval(", "exec(",
    "base64", "rot13", "hex encoded", "obfuscated",
    "ignore all", "new instructions", "you are now", "persona",
]

if is_phase_done("idf"):
    print("[RESUME] Loading IDF statistics from checkpoint ...")
    _idf_data = load_json_checkpoint("idf")
    idf_values = _idf_data["idf_values"]
    avg_doc_length = _idf_data["avg_doc_length"]
    avg_text_length = _idf_data["avg_text_length"]
    N = _idf_data["N"]
else:
    N = len(df)
    print(f"Computing corpus statistics for {N:,} documents ...")

    df_counts = {kw: 0 for kw in SPARSE_KEYWORDS}
    texts_lower = df["prompt_text"].str.lower()
    for kw in SPARSE_KEYWORDS:
        df_counts[kw] = int(texts_lower.str.contains(kw, regex=False).sum())

    idf_values = {}
    for kw in SPARSE_KEYWORDS:
        idf_values[kw] = math.log((N + 1) / (df_counts[kw] + 1)) + 1.0

    word_counts = df["prompt_text"].str.split().str.len()
    char_counts = df["prompt_text"].str.len()
    avg_doc_length = float(word_counts.mean())
    avg_text_length = float(char_counts.mean())

    save_json_checkpoint("idf", {
        "idf_values": idf_values,
        "avg_doc_length": avg_doc_length,
        "avg_text_length": avg_text_length,
        "N": N,
    })
    mark_phase_done("idf")
    upload_checkpoints()
    del texts_lower, word_counts, char_counts, df_counts
    gc.collect()

print(f"  N = {N:,}")
print(f"  Avg doc length: {avg_doc_length:.1f} words")
print(f"  Avg text length: {avg_text_length:.1f} chars")
print(f"  IDF range: [{min(idf_values.values()):.3f}, {max(idf_values.values()):.3f}]")

---
## Phase 3: Dense Embeddings (Sequential Dual-GPU)

Model: `BAAI/bge-large-en-v1.5` (1024-dim)

Encodes half the dataset on each GPU sequentially.

In [ ]:
from sentence_transformers import SentenceTransformer


def encode_on_gpu(texts_chunk, model_name, gpu_id, batch_size, normalize=True):
    device = torch.device(f"cuda:{gpu_id}")
    print(f"    Loading model on GPU {gpu_id} ...")
    model = SentenceTransformer(model_name, device=str(device))
    model.half()
    print(f"    Encoding {len(texts_chunk):,} texts on GPU {gpu_id} (batch_size={batch_size}, fp16=True) ...")
    embeddings = model.encode(
        texts_chunk,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=normalize,
        device=str(device),
        convert_to_numpy=True,
    )
    del model
    torch.cuda.empty_cache()
    gc.collect()
    return embeddings


if is_phase_done("embeddings"):
    print("[RESUME] Loading embeddings from checkpoint ...")
    dense_embeddings = load_npy_checkpoint("embeddings")
else:
    print(f"Loading dense model: {CFG['dense_model']}")
    t0 = time.time()

    texts = df["prompt_text"].tolist()
    n_texts = len(texts)
    half = n_texts // 2
    chunk_0 = texts[:half]
    chunk_1 = texts[half:]
    print(f"  Splitting {n_texts:,} texts: GPU 0 gets {len(chunk_0):,}, GPU 1 gets {len(chunk_1):,}")

    if num_gpus >= 2:
        print("  Phase 3a: Encoding chunk 0 on GPU 0 ...")
        emb_0 = encode_on_gpu(chunk_0, CFG["dense_model"], 0, CFG["embed_batch_size"])
        print(f"    GPU 0 done: {emb_0.shape}")
        gpu_memory_status()

        print("  Phase 3b: Encoding chunk 1 on GPU 1 ...")
        emb_1 = encode_on_gpu(chunk_1, CFG["dense_model"], 1, CFG["embed_batch_size"])
        print(f"    GPU 1 done: {emb_1.shape}")
        gpu_memory_status()

        dense_embeddings = np.concatenate([emb_0, emb_1], axis=0)
        del emb_0, emb_1
    else:
        gpu_id = 0
        print(f"  Using single GPU {gpu_id} ...")
        dense_embeddings = encode_on_gpu(texts, CFG["dense_model"], gpu_id, CFG["embed_batch_size"])

    dense_embeddings = dense_embeddings.astype(np.float16)

    save_npy_checkpoint("embeddings", dense_embeddings)
    mark_phase_done("embeddings")
    upload_checkpoints()

    elapsed = time.time() - t0
    print(f"  Time: {elapsed:.1f}s ({n_texts / elapsed:.0f} texts/sec)")
    clear_gpu()

print(f"  Embeddings: shape={dense_embeddings.shape}, dtype={dense_embeddings.dtype}")
gpu_memory_status()

---
## Phase 4: Category Centroids

In [ ]:
if is_phase_done("centroids"):
    print("[RESUME] Loading category centroids from checkpoint ...")
    category_centroids = load_json_checkpoint("centroids")
else:
    print("Computing category centroids ...")
    category_centroids = {}
    for category in df["attack_category"].unique():
        mask = df["attack_category"] == category
        cat_embeddings = dense_embeddings[mask]
        centroid = cat_embeddings.mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-8)
        category_centroids[category] = centroid.tolist()
        print(f"  {category:<35s} {int(mask.sum()):>8,} samples")

    save_json_checkpoint("centroids", category_centroids)
    mark_phase_done("centroids")
    upload_checkpoints()
    del cat_embeddings
    gc.collect()

print(f"  Total categories: {len(category_centroids)}")

---
## Phase 5: k-Means Clustering (FAISS-GPU)

In [ ]:
import faiss

if is_phase_done("kmeans"):
    print("[RESUME] Loading k-Means results from checkpoint ...")
    _km = load_json_checkpoint("kmeans")
    n_clusters = _km["n_clusters"]
    cluster_ids = np.array(_km["cluster_ids"], dtype=np.int64)
    cluster_centers = np.array(_km["cluster_centers"], dtype=np.float32)
else:
    print(f"Computing k-Means clustering (n_clusters={CFG['n_clusters']}) ...")
    t0 = time.time()

    n_clusters = min(CFG["n_clusters"], len(dense_embeddings) // 10)
    embeddings_f32 = dense_embeddings.astype(np.float32)
    d = embeddings_f32.shape[1]

    faiss_temp_bytes = CFG["faiss_temp_memory_mb"] * 1024 * 1024

    try:
        if num_gpus >= 2:
            print("  Using multi-GPU FAISS clustering ...")
            co = faiss.GpuMultipleClonerOptions()
            co.shard = True
            index_flat = faiss.IndexFlatL2(d)
            gpu_index = faiss.index_cpu_to_all_gpus(index_flat, co)
        elif num_gpus == 1:
            print("  Using single-GPU FAISS clustering ...")
            res = faiss.StandardGpuResources()
            res.setTempMemory(faiss_temp_bytes)
            index_flat = faiss.IndexFlatL2(d)
            gpu_index = faiss.index_cpu_to_gpu(res, 0, index_flat)
        else:
            raise RuntimeError("No GPU available")

        kmeans = faiss.Clustering(d, n_clusters)
        kmeans.niter = 20
        kmeans.gpu = True
        kmeans.train(embeddings_f32, gpu_index)

        _, cluster_ids = gpu_index.search(embeddings_f32, 1)
        cluster_ids = cluster_ids.flatten()

        cluster_centers = faiss.vector_to_array(kmeans.centroids).reshape(n_clusters, -1)

        del gpu_index, index_flat, kmeans
        clear_gpu()

    except Exception as e:
        print(f"  GPU clustering failed ({e}), falling back to CPU ...")
        index_cpu = faiss.IndexFlatL2(d)
        kmeans = faiss.Clustering(d, n_clusters)
        kmeans.niter = 20
        kmeans.gpu = False
        kmeans.train(embeddings_f32, index_cpu)

        index_cpu.add(embeddings_f32)
        _, cluster_ids = index_cpu.search(embeddings_f32, 1)
        cluster_ids = cluster_ids.flatten()

        cluster_centers = faiss.vector_to_array(kmeans.centroids).reshape(n_clusters, -1)
        del index_cpu, kmeans
        gc.collect()

    save_json_checkpoint("kmeans", {
        "n_clusters": int(n_clusters),
        "cluster_ids": cluster_ids.tolist(),
        "cluster_centers": cluster_centers.tolist(),
    })
    mark_phase_done("kmeans")
    upload_checkpoints()

    elapsed = time.time() - t0
    print(f"  Clustering done in {elapsed:.1f}s")

print(f"  Clusters: {n_clusters}")
bincounts = np.bincount(cluster_ids)
print(f"  Cluster size range: [{bincounts.min()}, {bincounts.max()}]")

---
## Phase 6: k-NN Graph (FAISS-GPU)

In [ ]:
if is_phase_done("knn"):
    print("[RESUME] Loading k-NN graph from checkpoint ...")
    _knn = load_json_checkpoint("knn")
    distances = np.array(_knn["distances"], dtype=np.float32)
    neighbor_indices = np.array(_knn["neighbor_indices"], dtype=np.int64)
else:
    print(f"Computing k-NN graph (k={CFG['n_neighbors']}) ...")
    t0 = time.time()

    k = CFG["n_neighbors"] + 1
    embeddings_f32 = dense_embeddings.astype(np.float32)
    d = embeddings_f32.shape[1]

    try:
        if num_gpus >= 2:
            print("  Using multi-GPU FAISS k-NN ...")
            co = faiss.GpuMultipleClonerOptions()
            co.shard = True
            index_flat = faiss.IndexFlatIP(d)
            gpu_index = faiss.index_cpu_to_all_gpus(index_flat, co)
        elif num_gpus == 1:
            print("  Using single-GPU FAISS k-NN ...")
            res = faiss.StandardGpuResources()
            res.setTempMemory(faiss_temp_bytes)
            index_flat = faiss.IndexFlatIP(d)
            gpu_index = faiss.index_cpu_to_gpu(res, 0, index_flat)
        else:
            raise RuntimeError("No GPU available")

        gpu_index.add(embeddings_f32)
        distances, neighbor_indices = gpu_index.search(embeddings_f32, k)

        del gpu_index, index_flat
        clear_gpu()

    except Exception as e:
        print(f"  GPU k-NN failed ({e}), falling back to CPU ...")
        index_cpu = faiss.IndexFlatIP(d)
        index_cpu.add(embeddings_f32)
        distances, neighbor_indices = index_cpu.search(embeddings_f32, k)
        del index_cpu
        gc.collect()

    save_json_checkpoint("knn", {
        "distances": distances.tolist(),
        "neighbor_indices": neighbor_indices.tolist(),
    })
    mark_phase_done("knn")
    upload_checkpoints()

    elapsed = time.time() - t0
    print(f"  k-NN graph computed in {elapsed:.1f}s")

print(f"  Mean distance to nearest neighbor: {distances[:, 1].mean():.4f}")
print(f"  Max distance to nearest neighbor:  {distances[:, 1].max():.4f}")

---
## Phase 7: Uniqueness Scores

In [ ]:
if is_phase_done("uniqueness"):
    print("[RESUME] Loading uniqueness scores from checkpoint ...")
    uniqueness = load_npy_checkpoint("uniqueness")
else:
    print("Computing uniqueness scores ...")

    mean_distances = distances[:, 1:].mean(axis=1)
    uniqueness = 1.0 / (mean_distances + 1e-8)
    uniqueness = (uniqueness - uniqueness.min()) / (uniqueness.max() - uniqueness.min() + 1e-8)

    save_npy_checkpoint("uniqueness", uniqueness)
    mark_phase_done("uniqueness")
    upload_checkpoints()

    del distances, mean_distances
    gc.collect()

print(f"  Uniqueness range: [{uniqueness.min():.4f}, {uniqueness.max():.4f}]")
print(f"  Mean uniqueness:  {uniqueness.mean():.4f}")

---
## Phase 8: Cross-Encoder Pre-Scoring (Sequential Dual-GPU)

Model: `cross-encoder/ms-marco-MiniLM-L-6-v2`

In [ ]:
from sentence_transformers import CrossEncoder

CATEGORY_DESCRIPTIONS = {
    "jailbreak": "prompt designed to bypass AI safety restrictions and make the model behave without guidelines",
    "direct_injection": "prompt that directly attempts to override system instructions or prepend new ones",
    "indirect_injection": "prompt embedded in external content that tries to manipulate AI behavior",
    "system_prompt_extraction": "prompt attempting to reveal or leak the system prompt or hidden instructions",
    "refusal_bypass": "prompt trying to make the model refuse less or comply with harmful requests",
    "benign_control": "legitimate non-malicious prompt used as a control sample",
}


def ce_score_on_gpu(pairs_chunk, model_name, gpu_id, batch_size):
    device = torch.device(f"cuda:{gpu_id}")
    print(f"    Loading cross-encoder on GPU {gpu_id} ...")
    model = CrossEncoder(model_name, device=str(device))
    model.half()
    print(f"    Scoring {len(pairs_chunk):,} pairs on GPU {gpu_id} (batch_size={batch_size}, fp16=True) ...")
    raw_scores = model.predict(pairs_chunk, batch_size=batch_size, show_progress_bar=True)
    scores = 1.0 / (1.0 + np.exp(-np.array(raw_scores, dtype=np.float32)))
    del model
    torch.cuda.empty_cache()
    gc.collect()
    return scores


if is_phase_done("crossencoder"):
    print("[RESUME] Loading cross-encoder scores from checkpoint ...")
    ce_scores = load_npy_checkpoint("crossencoder")
else:
    print("Building text-description pairs ...")
    cat_desc_map = df["attack_category"].map(CATEGORY_DESCRIPTIONS).fillna(df["attack_category"])
    pairs = list(zip(df["prompt_text"].tolist(), cat_desc_map.tolist()))
    print(f"  Total pairs: {len(pairs):,}")

    n_pairs = len(pairs)
    half_pairs = n_pairs // 2
    pairs_0 = pairs[:half_pairs]
    pairs_1 = pairs[half_pairs:]

    print(f"  Splitting: GPU 0 gets {len(pairs_0):,}, GPU 1 gets {len(pairs_1):,}")

    print(f"Loading cross-encoder: {CFG['cross_encoder_model']} ...")
    t0 = time.time()

    if num_gpus >= 2:
        print("  Phase 8a: Scoring chunk 0 on GPU 0 ...")
        scores_0 = ce_score_on_gpu(pairs_0, CFG["cross_encoder_model"], 0, CFG["ce_batch_size"])
        print(f"    GPU 0 done.")
        gpu_memory_status()

        print("  Phase 8b: Scoring chunk 1 on GPU 1 ...")
        scores_1 = ce_score_on_gpu(pairs_1, CFG["cross_encoder_model"], 1, CFG["ce_batch_size"])
        print(f"    GPU 1 done.")
        gpu_memory_status()

        ce_scores = np.concatenate([scores_0, scores_1])
        del scores_0, scores_1
    else:
        gpu_id = 0
        print(f"  Using single GPU {gpu_id} ...")
        ce_scores = ce_score_on_gpu(pairs, CFG["cross_encoder_model"], gpu_id, CFG["ce_batch_size"])

    save_npy_checkpoint("crossencoder", ce_scores)
    mark_phase_done("crossencoder")
    upload_checkpoints()

    elapsed = time.time() - t0
    print(f"  Time: {elapsed:.1f}s ({len(pairs) / elapsed:.0f} pairs/sec)")
    clear_gpu()

print(f"  Scores: range=[{ce_scores.min():.4f}, {ce_scores.max():.4f}], mean={ce_scores.mean():.4f}")

---
## Phase 9: Build Enhanced Payloads & Export

In [ ]:
if is_phase_done("export"):
    print("[RESUME] Export already complete — skipping.")
else:
    print("Building enhanced payloads ...")
    t0 = time.time()

    avg_text_len = float(df["prompt_text"].str.len().mean())
    text_lengths = df["prompt_text"].str.len().values
    length_norm = np.log1p(text_lengths.astype(np.float32)) / np.log1p(max(avg_text_len, 1))

    enhanced_df = df.copy()
    enhanced_df["cluster_id"] = cluster_ids
    enhanced_df["neighbor_ids"] = [idx[1:6].tolist() for idx in neighbor_indices]
    enhanced_df["uniqueness"] = uniqueness
    enhanced_df["cross_encoder_score"] = ce_scores
    enhanced_df["length_norm"] = length_norm
    enhanced_df["prompt_text"] = enhanced_df["prompt_text"].str[:CFG["max_text_len"]]

    elapsed = time.time() - t0
    print(f"  Payloads built in {elapsed:.1f}s")
    print(f"  Columns: {list(enhanced_df.columns)}")
    print(f"  Shape: {enhanced_df.shape}")

    corpus_meta = {
        "keyword_idf": idf_values,
        "total_documents": N,
        "avg_doc_length": avg_doc_length,
        "avg_text_length": avg_text_length,
        "category_centroids": category_centroids,
        "cluster_centers": cluster_centers.tolist(),
        "scoring_weights": {
            "dense": 0.40,
            "sparse_idf": 0.20,
            "centroid": 0.15,
            "cross_encoder": 0.15,
            "uniqueness": 0.05,
            "length_norm": 0.05,
        },
    }

    meta_path = os.path.join(CFG["out_dir"], "corpus_meta.json")
    with open(meta_path, "w") as f:
        json.dump(corpus_meta, f, indent=2)
    print(f"Saved corpus_meta.json ({os.path.getsize(meta_path) / 1024:.0f} KB)")

    payload_path = os.path.join(CFG["out_dir"], "enhanced_payloads.parquet")
    enhanced_df.to_parquet(payload_path, engine="pyarrow", index=False)
    print(f"Saved enhanced_payloads.parquet ({os.path.getsize(payload_path) / 1e6:.1f} MB)")

    emb_path = os.path.join(CFG["out_dir"], "dense_embeddings_float16.npy")
    np.save(emb_path, dense_embeddings.astype(np.float16))
    print(f"Saved dense_embeddings_float16.npy ({os.path.getsize(emb_path) / 1e6:.1f} MB)")

    mark_phase_done("export")
    upload_checkpoints()
    print(f"\nAll files saved to {CFG['out_dir']}")

In [ ]:
print("=" * 60)
print("EXPORT SUMMARY")
print("=" * 60)

out_dir = CFG["out_dir"]
for f in sorted(os.listdir(out_dir)):
    if f.startswith("."):
        continue
    fpath = os.path.join(out_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        if size > 1e6:
            print(f"  {f:<45s} {size/1e6:>8.1f} MB")
        else:
            print(f"  {f:<45s} {size/1024:>8.0f} KB")

print(f"\nDataset: {N:,} documents")
print(f"Embedding dim: {dense_embeddings.shape[1]}")
print(f"Clusters: {n_clusters}")
print(f"Neighbors per node: {CFG['n_neighbors']}")
print(f"GPUs used: {num_gpus}")
print("=" * 60)

print(f"\nCheckpoint status:")
for pn in PHASE_NAMES:
    status = "DONE" if is_phase_done(pn) else "pending"
    print(f"  {pn:<20s} {status}")

---
## How Resumption Works

**What you do:** Nothing. Just re-run all cells.

**What the notebook does automatically:**
1. Detects your Kaggle username from `kaggle.json` (already authenticated on Kaggle)
2. Creates a Kaggle Dataset `guardrailer-checkpoints` if it doesn't exist
3. Downloads previous checkpoints from that Dataset
4. Skips completed phases, recomputes pending ones
5. After each phase, uploads checkpoints to the Dataset (persistent storage)
6. If session is killed, `atexit`/signal handlers flush + upload before exit

| Scenario | What happens |
|---|---|
| Session killed mid-run | Watchdog saves + uploads. Re-run all cells to resume. |
| New session, same account | Auto-downloads, resumes from last completed phase. |
| New session, different account | Set `CFG['ckpt_dataset']` to the original Dataset slug. |
| Want to start fresh | Delete the Kaggle Dataset from your account. |

### Checkpoint files
All stored in `/kaggle/working/guardrailer_output/.checkpoints/` (local) and synced to `your_username/guardrailer-checkpoints` (Kaggle Dataset).

---
## Download Instructions

After the notebook completes, run the cell below to create a single zip file for easy download.

Then download `guardrailer_all_outputs.zip` from the Kaggle output panel.

In [ ]:
import zipfile, shutil

out_dir = CFG["out_dir"]

# Split dense embeddings into 100MB chunks for Kaggle download limit
emb_path = os.path.join(out_dir, "dense_embeddings_float16.npy")
if os.path.exists(emb_path):
    emb = np.load(emb_path)
    chunk_size_mb = 80
    bytes_per_sample = emb.nbytes // len(emb)
    samples_per_chunk = max(1, (chunk_size_mb * 1024 * 1024) // bytes_per_sample)
    n_chunks = (len(emb) + samples_per_chunk - 1) // samples_per_chunk

    print(f"Splitting embeddings ({emb.nbytes / 1e6:.0f} MB) into {n_chunks} chunks ...")
    for i in range(n_chunks):
        start = i * samples_per_chunk
        end = min(start + samples_per_chunk, len(emb))
        chunk_path = os.path.join(out_dir, f"dense_embeddings_part_{i}.npy")
        np.save(chunk_path, emb[start:end])
        print(f"  Part {i}: samples {start}-{end} ({os.path.getsize(chunk_path) / 1e6:.1f} MB)")

    # Save shape info for reassembly
    meta = {"total_samples": len(emb), "dim": emb.shape[1], "dtype": str(emb.dtype), "n_chunks": n_chunks}
    with open(os.path.join(out_dir, "embeddings_meta.json"), "w") as f:
        json.dump(meta, f)
    del emb
    gc.collect()
else:
    print(f"WARNING: {emb_path} not found")

# Pack all output files
zip_path = os.path.join(out_dir, "guardrailer_all_outputs.zip")
files_to_pack = [
    "corpus_meta.json",
    "enhanced_payloads.parquet",
    "embeddings_meta.json",
]
# Add chunk files
for f in sorted(os.listdir(out_dir)):
    if f.startswith("dense_embeddings_part_") and f.endswith(".npy"):
        files_to_pack.append(f)

print(f"\nPacking into {zip_path} ...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in files_to_pack:
        fpath = os.path.join(out_dir, fname)
        if os.path.exists(fpath):
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"  {fname} ({size_mb:.1f} MB)")
            zf.write(fpath, fname)

zip_size = os.path.getsize(zip_path) / 1e6
print(f"\nDone: {zip_size:.1f} MB total")

In [ ]:
import subprocess

# Push zip to Kaggle Dataset (no file size limit on dataset downloads)
CKPT_SLUG = f"{CFG['kaggle_username']}/{CFG['ckpt_dataset']}" if 'KAGGLE_USER' in dir() and KAGGLE_USER else ""

if not CKPT_SLUG:
    print("[ERROR] Kaggle user not detected. Upload manually.")
else:
    print(f"Uploading to {CKPT_SLUG} ...")
    print("After upload, go to:")
    print(f"  https://www.kaggle.com/datasets/{CKPT_SLUG}")
    print("  Click Files tab -> Download guardrailer_all_outputs.zip")
    print()

    result = subprocess.run(
        ["kaggle", "datasets", "version",
         "-p", out_dir,
         "-m", f"output zip {time.strftime('%Y-%m-%d %H:%M:%S')}",
         "-t", CKPT_SLUG],
        capture_output=True, text=True, timeout=600
    )
    if result.returncode == 0:
        print(f"SUCCESS! Download from: https://www.kaggle.com/datasets/{CKPT_SLUG}")
    else:
        print(f"Upload failed: {result.stderr[:300]}")
        print("\nManual download: click the zip filename in Kaggle output panel.")